# Neural Network from Scratch - Code Context for NotebookLM

This notebook implements a simple feedforward Neural Network from scratch using only `numpy`. 

## Architecture
- **Input Layer**: 784 nodes (for 28x28 pixel images).
- **Hidden Layer**: 128 neurons with **ReLU** activation.
- **Output Layer**: 10 neurons (for digits 0-9) with **Softmax** activation.

## Key Components
- **Initialization**: Uses He Initialization for weights (`W1`, `W2`).
- **Forward Propagation (`forward_prop`)**: Computes `z1`, `A1` (ReLU), `z2`, `A2` (Softmax).
- **Loss Function (`compute_loss`)**: Uses Cross-Entropy Loss.
- **Backpropagation (`back_prop`)**: Computes gradients to update weights.
- **Training (`gradient_descent`)**: Uses Mini-Batch Gradient Descent.

This context is provided to help NotebookLM understand the structure of the code.


In [2]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib


  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp314-cp314-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl (12.4 MB)
Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.3 MB 13.8 MB/s eta 0:00:01
   ---------------------- ----------------- 4.7/8.3 MB 12.9 MB/s eta 0:00:01
   ----------------------------------- ---- 7.3/8.3 MB 12.6 MB/s eta 0:00:01
   ---

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\sawan\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 

In [4]:
data = pd.read_csv(r"C:\Users\sawan\Downloads\digit-recognizer\train.csv")


In [5]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
data = np.array(data)\nm, n = data.shape\nnp.random.shuffle(data)\n\ndata_dev = data[0:1000].T\n\ny_dev = data_dev[0]\nx_dev = data_dev[1:n] / 255.\n\ndata_train = data[1000:m].T\ny_train = data_train[0]\nx_train = data_train[1:n] / 255.

In [ ]:
def init_params(input_size=784, hidden1=128, hidden2=64, output_size=10):
    # He Initialization for 3 layers
    W1 = np.random.randn(hidden1, input_size) * np.sqrt(2. / input_size)
    B1 = np.zeros((hidden1, 1))
    W2 = np.random.randn(hidden2, hidden1) * np.sqrt(2. / hidden1)
    B2 = np.zeros((hidden2, 1))
    W3 = np.random.randn(output_size, hidden2) * np.sqrt(2. / hidden2)
    B3 = np.zeros((output_size, 1))
    return W1, B1, W2, B2, W3, B3

def init_adam(W1, B1, W2, B2, W3, B3):
    # Initializes momentums for Adam optimizer
    V = {"dW1": np.zeros(W1.shape), "dB1": np.zeros(B1.shape),
         "dW2": np.zeros(W2.shape), "dB2": np.zeros(B2.shape),
         "dW3": np.zeros(W3.shape), "dB3": np.zeros(B3.shape)}
    S = {"dW1": np.zeros(W1.shape), "dB1": np.zeros(B1.shape),
         "dW2": np.zeros(W2.shape), "dB2": np.zeros(B2.shape),
         "dW3": np.zeros(W3.shape), "dB3": np.zeros(B3.shape)}
    return V, S

def RElu(z):
    return np.maximum(0, z)

def softmax(z):
    z_shifted = z - np.max(z, axis=0, keepdims=True)
    return np.exp(z_shifted) / np.sum(np.exp(z_shifted), axis=0, keepdims=True)

def forward_prop(W1, B1, W2, B2, W3, B3, X, keep_prob=1.0):
    z1 = W1.dot(X) + B1
    A1 = RElu(z1)
    
    # Dropout Layer 1
    D1 = np.random.rand(A1.shape[0], A1.shape[1]) < keep_prob
    A1 = (A1 * D1) / keep_prob

    z2 = W2.dot(A1) + B2
    A2 = RElu(z2)
    
    # Dropout Layer 2
    D2 = np.random.rand(A2.shape[0], A2.shape[1]) < keep_prob
    A2 = (A2 * D2) / keep_prob

    z3 = W3.dot(A2) + B3
    A3 = softmax(z3)
    
    return z1, A1, D1, z2, A2, D2, z3, A3

def compute_loss(A3, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)
    epsilon = 1e-15 # To prevent log(0)
    loss = -np.sum(one_hot_Y * np.log(A3 + epsilon)) / m
    return loss

def one_hot(y):
    one_hot_y = np.zeros((y.size, 10))
    one_hot_y[np.arange(y.size), y] = 1
    return one_hot_y.T

def derivative_RElu(z):
    return z > 0

def back_prop(z1, A1, D1, z2, A2, D2, z3, A3, W1, W2, W3, X, y, keep_prob=1.0):
    m = y.size
    one_hot_y = one_hot(y)
    
    # Output Layer
    dz3 = A3 - one_hot_y
    dW3 = 1/m * dz3.dot(A2.T)
    dB3 = 1/m * np.sum(dz3, axis=1, keepdims=True)
    
    # Hidden Layer 2
    dz2 = W3.T.dot(dz3) * derivative_RElu(z2)
    dz2 = (dz2 * D2) / keep_prob # Apply Dropout mask
    dW2 = 1/m * dz2.dot(A1.T)
    dB2 = 1/m * np.sum(dz2, axis=1, keepdims=True)
    
    # Hidden Layer 1
    dz1 = W2.T.dot(dz2) * derivative_RElu(z1)
    dz1 = (dz1 * D1) / keep_prob # Apply Dropout mask
    dW1 = 1/m * dz1.dot(X.T)
    dB1 = 1/m * np.sum(dz1, axis=1, keepdims=True)
    
    return dW1, dB1, dW2, dB2, dW3, dB3

def update_params(W1, B1, W2, B2, W3, B3, dW1, dB1, dW2, dB2, dW3, dB3, V, S, t, alpha, beta1=0.9, beta2=0.999, epsilon=1e-8):
    # Update momentums
    grads = {"dW1": dW1, "dB1": dB1, "dW2": dW2, "dB2": dB2, "dW3": dW3, "dB3": dB3}
    params = {"W1": W1, "B1": B1, "W2": W2, "B2": B2, "W3": W3, "B3": B3}
    
    for key in grads:
        V[key] = beta1 * V[key] + (1 - beta1) * grads[key]
        S[key] = beta2 * S[key] + (1 - beta2) * (grads[key] ** 2)
        
        # Bias correction
        v_corrected = V[key] / (1 - beta1**t)
        s_corrected = S[key] / (1 - beta2**t)
        
        # Update param
        params[key] = params[key] - alpha * (v_corrected / (np.sqrt(s_corrected) + epsilon))
        
    return params["W1"], params["B1"], params["W2"], params["B2"], params["W3"], params["B3"], V, S


(784,)

In [ ]:
def get_predictions(A3):
    return np.argmax(A3, 0)

def get_accuracy(predictions, y):
    return np.sum(predictions == y) / y.size

def gradient_descent(X, Y, alpha, epochs, batch_size=128, keep_prob=0.8):
    # Initialize 3 layers
    W1, B1, W2, B2, W3, B3 = init_params(784, 128, 64, 10)
    V, S = init_adam(W1, B1, W2, B2, W3, B3)
    t = 0 # Adam iteration counter
    
    m = X.shape[1] 
    
    for epoch in range(epochs):
        permutation = np.random.permutation(m)
        X_shuffled = X[:, permutation]
        Y_shuffled = Y[permutation]
        
        for i in range(0, m, batch_size):
            X_batch = X_shuffled[:, i:i+batch_size]
            Y_batch = Y_shuffled[i:i+batch_size]
            
            # Forward pass with Dropout
            z1, A1, D1, z2, A2, D2, z3, A3 = forward_prop(W1, B1, W2, B2, W3, B3, X_batch, keep_prob)
            
            # Backward pass
            dW1, dB1, dW2, dB2, dW3, dB3 = back_prop(z1, A1, D1, z2, A2, D2, z3, A3, W1, W2, W3, X_batch, Y_batch, keep_prob)
            
            # Update params with Adam
            t += 1
            W1, B1, W2, B2, W3, B3, V, S = update_params(
                W1, B1, W2, B2, W3, B3, 
                dW1, dB1, dW2, dB2, dW3, dB3, 
                V, S, t, alpha
            )
        
        if epoch % 5 == 0:
            # Eval without dropout (keep_prob=1.0)
            _, _, _, _, _, _, _, A3_full = forward_prop(W1, B1, W2, B2, W3, B3, X, keep_prob=1.0)
            predictions = get_predictions(A3_full)
            accuracy = get_accuracy(predictions, Y)
            loss = compute_loss(A3_full, Y)
            print(f"Epoch: {epoch} | Loss: {loss:.4f} | Accuracy: {accuracy:.4f}")
            
    return W1, B1, W2, B2, W3, B3


In [ ]:
# Train for 20 epochs (Adam is much faster than standard GD)
# Learning rate is smaller (0.001) for Adam
W1, B1, W2, B2, W3, B3 = gradient_descent(x_train, y_train, alpha=0.001, epochs=20, batch_size=128, keep_prob=0.8)

In [ ]:
import matplotlib.pyplot as plt

# Grab a single image from the validation set
image_index = 0
my_image = x_dev[:, image_index].reshape(784, 1)
true_label = y_dev[image_index]

# Pass through the 3-layer trained network (keep_prob=1.0 for testing)
_, _, _, _, _, _, _, A3 = forward_prop(W1, B1, W2, B2, W3, B3, my_image, keep_prob=1.0)
prediction = get_predictions(A3)[0]

print(f"The model predicts this is a: {prediction}")
print(f"The actual label is: {true_label}")

image_grid = my_image.reshape(28, 28)
plt.imshow(image_grid, cmap='gray')
plt.title(f"Prediction: {prediction} | Actual: {true_label}")
plt.axis('off')
plt.show()

In [ ]:
# Save your trained 3-layer model weights
np.savez("my_trained_mnist_model_v2.npz", W1=W1, B1=B1, W2=W2, B2=B2, W3=W3, B3=B3)
print("Model saved successfully!")

In [ ]:
# Load the model back in
saved_model = np.load("my_trained_mnist_model_v2.npz")
W1 = saved_model['W1']
B1 = saved_model['B1']
W2 = saved_model['W2']
B2 = saved_model['B2']
W3 = saved_model['W3']
B3 = saved_model['B3']
print("Model loaded successfully!")